# learner

> Pre-training, fine-tuning, and probing for PRAGMA

Built on fastai. Supports masked modelling pre-training, LoRA fine-tuning, and L-BFGS linear probing.

In [ ]:
#| default_exp learner

In [ ]:
#| export
import torch, torch.nn as nn, torch.nn.functional as F
from fastai.data.all import *
from fastai.learner import *
from fastai.optimizer import *
from fastai.callback.schedule import *

## MLM Dataset

Wraps per-entity tokenized dicts into a fastai-compatible dataset with random masking.

During pre-training, 15% of value tokens are randomly replaced with a `[MASK]` token (ID=1). The model must predict the original value IDs at those positions — this is the masked language modelling (MLM) objective from the paper (§2.3.5). The dataset returns all fields needed by the three-block architecture: keys, masked values, times, masks, profile_mask, and calendar features.

In [ ]:
#| export
class MLMDataset:
    """Wraps a PRAGMADataset for masked language modelling (MLM) pre-training.
    Randomly masks value tokens with probability mask_prob; the model must predict
    the original value IDs at masked positions (paper §2.3.5)."""
    def __init__(self, pragma_ds, mask_prob=0.15):
        self.tokens = pragma_ds.tokenize()
        self.entities = list(self.tokens.keys())
        self.mask_id = 1  # token ID used for masked positions
        self.mask_prob = mask_prob

    def __len__(self): return len(self.entities)

    def __getitem__(self, i):
        t = self.tokens[self.entities[i]]
        k, v, ti, m = t['keys'], t['vals'], t['times'], t['masks']
        pm = t.get('profile_mask')  # True = profile token, False = event token
        cal = t.get('calendar')     # (L, 3) calendar features
        n = k.shape[0]
        if n == 0:
            empty_mask = torch.zeros(0, dtype=torch.bool)
            return k, v, ti, m, empty_mask, k, torch.zeros(0, dtype=torch.long), pm, cal
        # Random mask: 15% of tokens are masked (paper §2.3.5)
        mask = torch.rand(n) < self.mask_prob
        targets = v[mask].clone()           # original value IDs at masked positions
        v_masked = v.clone()
        v_masked[mask] = self.mask_id        # replace with [MASK] token
        # Returns: keys, masked_vals, times, masks, mask, targets, mask, profile_mask, calendar
        return k, v_masked, ti, m, mask, targets, mask, pm, cal

In [ ]:
from fastcore.test import test_eq

# Create dummy PRAGMADataset with known tokens
class FakeDS:
    def tokenize(self):
        k = torch.arange(5)
        v = torch.randint(0, 50, (5,))
        t = torch.ones(5)
        m = torch.ones(5, dtype=torch.bool)
        pm = torch.tensor([True, True, False, False, False])
        cal = torch.randn(5, 3)
        return {0: dict(keys=k, vals=v, times=t, masks=m, profile_mask=pm, calendar=cal),
                1: dict(keys=k, vals=v, times=t, masks=m, profile_mask=pm, calendar=cal)}

mlm_ds = MLMDataset(FakeDS(), mask_prob=0.5)
test_eq(len(mlm_ds), 2)
k, v, t, m, mk, tgs, tgt_mk, pm, cal = mlm_ds[0]
test_eq(k.shape, (5,))
test_eq(v.shape, (5,))
assert tgs.numel() > 0  # some masked
test_eq(pm.shape, (5,))
test_eq(cal.shape, (5, 3))
print('MLMDataset: ok')

MLMDataset: ok


In [ ]:
#| export
def _collate_mlm(batch):
    """Collate variable-length MLM samples into batched tensors with padding.
    Pads all sequences to max_len in the batch; padding mask (True = real, False = pad)
    ensures attention ignores padded positions. Returns (x_batch, y_batch) where
    x_batch = (keys, vals, times, masks, profile_masks, calendars) and
    y_batch = (targets, target_masks) for cross-entropy loss."""
    ks, vs, ts, ms, mks, tgs, tgt_masks, pms, cals = zip(*batch)
    max_len = max(k.shape[0] for k in ks)

    def _pad(t, max_len, pad_val=0):
        """Pad tensor t along dim 0 to max_len with pad_val."""
        n = t.shape[0]
        if n == max_len: return t
        pad_shape = (max_len - n,) + t.shape[1:]
        return torch.cat([t, torch.full(pad_shape, pad_val, dtype=t.dtype, device=t.device)])

    # Pad and stack 1D tensors (keys, vals, times)
    keys = torch.stack([_pad(k, max_len) for k in ks])
    vals = torch.stack([_pad(v, max_len) for v in vs])
    times = torch.stack([_pad(t, max_len) for t in ts])
    # Padding mask: True for real tokens, False for padding (used as key_padding_mask)
    masks = torch.stack([torch.cat([m, torch.zeros(max_len - m.shape[0], dtype=torch.bool)]) for m in ms])
    # Profile mask: True = profile token, False = event or padding
    profile_masks = None
    if pms[0] is not None:
        profile_masks = torch.stack([torch.cat([pm, torch.zeros(max_len - pm.shape[0], dtype=torch.bool)]) for pm in pms])
    # Calendar features: (B, max_len, 3), padded with zeros
    calendars = None
    if cals[0] is not None:
        calendars = torch.stack([torch.cat([cal, torch.zeros(max_len - cal.shape[0], cal.shape[1])]) for cal in cals])

    # Targets: flatten all masked token values across the batch
    targets = torch.cat(tgs)
    # Target masks: (B, max_len) — True at masked positions, padded with False
    target_masks = torch.cat([torch.cat([tm, torch.zeros(max_len - tm.shape[0], dtype=torch.bool)]) for tm in tgt_masks])
    return (keys, vals, times, masks, profile_masks, calendars), (targets, target_masks)

In [ ]:
# Test _collate_mlm with variable-length sequences + profile_mask + calendar
ks = [torch.randint(0, 10, (3,)), torch.randint(0, 10, (5,))]
vs = [torch.randint(0, 10, (3,)), torch.randint(0, 10, (5,))]
ts = [torch.ones(3), torch.ones(5)]
ms = [torch.ones(3, dtype=torch.bool), torch.ones(5, dtype=torch.bool)]
mks = [torch.tensor([True, True, False]), torch.tensor([True, False, False, True, True])]
tgs = [torch.tensor([1,2]), torch.tensor([3,4,5])]
tgt_masks = [torch.tensor([True, True, False]), torch.tensor([True, False, False, True, True])]
pms = [torch.tensor([True, False, False]), torch.tensor([True, True, False, False, False])]
cals = [torch.randn(3, 3), torch.randn(5, 3)]

batch = list(zip(ks, vs, ts, ms, mks, tgs, tgt_masks, pms, cals))
(x_out, y_out) = _collate_mlm(batch)
keys, vals, times, masks, profile_masks, calendars = x_out
targets, target_masks = y_out
test_eq(keys.shape, (2, 5))       # padded to max_len=5
test_eq(vals.shape, (2, 5))
test_eq(masks.shape, (2, 5))
test_eq(profile_masks.shape, (2, 5))
test_eq(calendars.shape, (2, 5, 3))
test_eq(targets.shape, (5,))       # 2+3 targets
test_eq(target_masks.shape, (2, 5))
print('_collate_mlm: ok')

AssertionError: ==:
torch.Size([10])
(2, 5)

## LoRA

Low-Rank Adaptation (Hu et al., 2021) for parameter-efficient fine-tuning.

Instead of updating all backbone weights, LoRA adds a trainable low-rank decomposition `A @ B` to each linear layer. Only A and B are trained (rank × d_model parameters each), while the original weights are frozen. This reduces trainable parameters by ~100x while achieving comparable performance.

`apply_lora` wraps `nn.Linear` modules in `TransformerBlock` (QKV projections and MLP) with `LoRALinear`. It deliberately skips `mlm_head`, `cal_mlp`, and embedding layers — these are either task-specific heads or don't benefit from low-rank adaptation.

In [ ]:
#| export
class LoRALinear(nn.Module):
    """Low-Rank Adaptation wrapper (Hu et al., 2021). Adds a trainable low-rank
    delta W*x to a frozen linear layer: output = W*x + (x @ A @ B) * scale.
    A is init with small random values; B is init to zero (so delta starts at 0)."""
    def __init__(self, linear, rank=8, alpha=8):
        super().__init__()
        self.linear = linear  # frozen base layer (original weights preserved)
        self.rank = rank
        self.scale = alpha / rank  # scaling factor: alpha/rank controls LoRA contribution
        # Low-rank decomposition: d_model → rank → d_model
        self.lora_A = nn.Parameter(torch.randn(linear.in_features, rank) * 0.02)  # small random init
        self.lora_B = nn.Parameter(torch.zeros(rank, linear.out_features))        # zero init (no initial delta)

    def forward(self, x):
        return self.linear(x) + (x @ self.lora_A @ self.lora_B) * self.scale

# Note: apply_lora is defined in the Learner Factory section below,
# where it correctly skips mlm_head, cal_mlp, and embedding layers.

In [ ]:
# Test LoRA
lin = nn.Linear(16, 8)
orig_out = lin(torch.randn(2, 16))

lora_lin = LoRALinear(lin, rank=4, alpha=8)
x = torch.randn(2, 16)
out = lora_lin(x)
test_eq(out.shape, (2, 8))
assert not torch.allclose(out, orig_out, atol=1e-4)  # LoRA adds delta

# Test apply_lora
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)
        self.emb = nn.Embedding(10, 4)

m = TinyModel()
n_before = sum(p.numel() for p in m.parameters())
m = apply_lora(m, rank=4, alpha=8)
n_after = sum(p.numel() for p in m.parameters())
assert n_after > n_before  # LoRA added params
print('LoRA: ok')
# Test apply_lora skips mlm_head
class TinyPRAGMA(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)
        self.mlm_head = nn.Sequential(nn.Linear(4, 4), nn.GELU(), nn.Linear(4, 10))
        self.key_emb = nn.Embedding(10, 4)

m2 = TinyPRAGMA()
m2 = apply_lora(m2, rank=4, alpha=8)
assert isinstance(m2.fc, LoRALinear)  # fc gets LoRA
assert not isinstance(m2.mlm_head[0], LoRALinear)  # mlm_head skipped
print('apply_lora skip: ok')

NameError: name 'apply_lora' is not defined

## Learner Factory

Creates a `PRAGMALearner` wrapping a fastai `Learner` with PRAGMA-specific methods.

- **Pre-training**: `learner(dls, model)` with `head=None` → uses `MLMLoss` (cross-entropy on masked value predictions)
- **Fine-tuning**: `learner(dls, model, head='classification')` → adds a linear head and applies LoRA to the backbone
- **Probing**: `PRAGMALearner.probe()` → extracts frozen embeddings and fits an L-BFGS logistic regression (paper §3.2)
- **Persistence**: `save(name)` / `load(dls, name, size='s')` for checkpointing

In [ ]:
#| export
def mlm_loss(preds, targs):
    "Functional MLM loss for direct use."
    targets, tmask = targs
    if targets.numel() == 0: return torch.tensor(0.0, requires_grad=True)
    return F.cross_entropy(preds[tmask], targets)

class MLMLoss:
    "Loss for MLM pre-training. Expects model output from forward(return_mlm=True): (hist_in, logits)."
    def __call__(self, out, targs):
        targets, tmask = targs
        if targets.numel() == 0: return torch.tensor(0.0, requires_grad=True)
        logits = out[1] if isinstance(out, tuple) else out  # logits from return_mlm
        preds = logits[tmask]
        return F.cross_entropy(preds, targets)

class PRAGMALearner:
    """Wraps a fastai Learner with PRAGMA-specific methods for pre-training,
    fine-tuning (LoRA), probing (L-BFGS linear), and model persistence."""
    def __init__(self, learn): self.learn = learn
    def fit(self, n): self.learn.fit(n)
    def fine_tune(self, n, lr=1e-3):
        """Apply LoRA to backbone then fine-tune for n epochs at lr."""
        apply_lora(self.learn.model)
        self.learn.fit(n, lr)
    def probe(self, n=1, method='lbfgs'):
        """Linear probe: extract embeddings, fit logistic regression with L-BFGS.
        n controls max_iter (n*1000). No backbone training — backbone is frozen."""
        from scipy.optimize import minimize
        from sklearn.linear_model import LogisticRegression
        # Extract embeddings by running backbone in eval mode
        model = self.learn.model
        model.eval()
        embs, labels = [], []
        with torch.no_grad():
            for xb, yb in self.learn.dls.train:
                h = model(*xb[:4])  # keys, vals, times, masks (skip profile_mask/calendar)
                embs.append(h.mean(dim=1))  # mean pool over sequence
                labels.append(yb)
        X = torch.cat(embs).numpy()
        y = torch.cat(labels).numpy()
        self.probe_model = LogisticRegression(method='lbfgs', max_iter=n*1000)
        self.probe_model.fit(X, y)
    def save(self, name): torch.save(self.learn.model.state_dict(), f'{name}.pth')
    @classmethod
    def load(cls, dls, name, **kwargs):
        """Load a saved model. kwargs passed to PRAGMA.load (e.g. size='s')."""
        model = PRAGMA.load(**kwargs)
        model.load_state_dict(torch.load(f'{name}.pth'))
        learn = Learner(dls, model, loss_func=MLMLoss(), opt_func=Adam)
        return cls(learn)

def learner(dls, model, head=None, lora_rank=8, lora_alpha=8):
    """Create a PRAGMALearner. If head='classification' or 'regression', adds a
    classification/regression head and applies LoRA for fine-tuning.
    If head=None, uses MLMLoss for pre-training."""
    loss_func = MLMLoss() if head is None else (nn.CrossEntropyLoss() if head == 'classification' else nn.MSELoss())
    learn = Learner(dls, model, loss_func=loss_func, opt_func=Adam)
    if head:
        n_out = dls.c if hasattr(dls, 'c') else 1
        model.head = nn.Linear(model.d_model, n_out)
        apply_lora(model, lora_rank, lora_alpha)
    return PRAGMALearner(learn)

def apply_lora(model, rank=8, alpha=8):
    """Apply LoRA to QKV projections and MLP layers only (not mlm_head or embeddings).
    Replaces nn.Linear modules in TransformerBlock.attn and TransformerBlock.mlp
    with LoRALinear wrappers. Skips mlm_head, cal_mlp, key_emb, val_emb."""
    skip = {'mlm_head', 'cal_mlp', 'key_emb', 'val_emb'}
    for name, mod in list(model.named_modules()):
        if not isinstance(mod, nn.Linear): continue
        # Skip if inside mlm_head or cal_mlp
        if any(s in name for s in skip): continue
        # Navigate to parent module and replace the linear with LoRALinear
        parent = model
        for part in name.split('.')[:-1]: parent = getattr(parent, part)
        setattr(parent, name.split('.')[-1], LoRALinear(mod, rank, alpha))
    return model

In [ ]:
# Test MLMLoss with three-block return_mlm path
model = PRAGMA.load('s')
mlm_ds = MLMDataset(FakeDS())

# MLMLoss: model returns (hist_in, logits) when return_mlm=True
loss_fn = MLMLoss()
pm = torch.tensor([[True, True, False, False, False]])
cal = torch.randn(1, 5, 3)
masks = torch.ones(1, 5, dtype=torch.bool)
out = model(torch.randint(0, 60, (1, 5)), torch.randint(0, 100, (1, 5)),
            torch.rand(1, 5), masks=masks, profile_mask=pm, calendar=cal, return_mlm=True)
targs = (torch.tensor([1, 2, 3]), torch.tensor([[True, True, True, False, False]]))
loss = loss_fn(out, targs)
assert loss.item() > 0
print(f'MLMLoss: ok (loss={loss.item():.3f})')

# PRAGMALearner save/load roundtrip
class FakeDls:
    def __init__(self): self.c = 2
    train = []
plr = PRAGMALearner.__new__(PRAGMALearner)
plr.learn = type('L', (), {'model': model})()
plr.save('/tmp/_test_fp')
print('save: ok')
loaded = PRAGMALearner.load(FakeDls(), '/tmp/_test_fp', size='s')
print('load: ok')
Path('/tmp/_test_fp.pth').unlink()

NameError: name 'PRAGMA' is not defined